# Fit GEAGS model to experimental data

In [58]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.embed import file_html
from IPython.display import HTML, display
from bokeh.resources import CDN, INLINE
import html as html_lib
import scipy.integrate as scint
from lmfit import minimize, fit_report, Parameters

# Works when launched from the repository root or any directory beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Yuhang/plate_reader_analysis_diya_kirill_3.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Launch this notebook from within the CcaSolitaiRe repository.')
DATA_DIR = ROOT / 'Yuhang/Data'

In [59]:
# DEFINE GEAGS ODE model

# ode calculation function
def dxdt(t, x, params, log_growth_params = None, light = 'dark'):
    """
    ODE calculation function
    
    Args:
        t,x: time and state variables
        params: model parameters
        log_growth_params: logistic growth parameters
        light: light condition ('dark', 'green', 'red')
    Returns:
        dxdt: derivative of state variables
    """

    # Determine the light light
    if light.lower() == 'green':
        red, green = (0, 1)
    elif light.lower() == 'red':
        red, green = (1, 0)
    else:
        red, green = (0, 0)

    # Unpack variables
    S = x[0]    # CcaS protein
    Sp = x[1]   # phosphorylated CcaS protein
    R = x[2]    # CcaR protein
    Rp = x[3]   # phosphorylated CcaS protein
    Ac = x[4]   # active CcaR dimer
    M = x[5]    # CcaSR-regulated mRNA (sfGFP)
    C_tic = x[6]    # sfGFP mRNA-ribosome translation initiation complex conc.
    P = x[7]        # nascent sfGFP protein
    Pm = x[8]       # mature sfGFP protein
    N = x[9]        # culture OD


    # Unpack parameters from an lmfit parameters object
    p = params.valuesdict()
    k_green = p['k_green'] * green # 1/min
    k_red = p['k_red'] * red # 1/min
    b_green = p['b_green'] # 1/min
    b_red = p['b_red'] # 1/min
    k_sp_b = p['k_sp_b'] # 1/min
    k_sp_u = p['k_sp_u'] # 1/min
    k_rp_b = p['k_rp_b'] # 1/min
    k_rp_u = p['k_rp_u'] # 1/min

    beta = p['beta'] # nM/min
    l0 = p['l0'] # leak coefficient
    Kc = p['Kc'] # nM
    d_m = p['d_m'] # 1/min
    k_tli_b = p['k_tli_b'] # 1/min
    k_tli_u = p['k_tli_u'] # 1/min
    k_tl = p['k_tl'] # 1/min
    d_p = p['d_p'] # 1/min
    k_fold = p['k_fold'] # 1/min
    b_fold = p['b_fold'] # 1/min
    n_delta = 5.5

    n_gamma = p['n_gamma']
    R_max = p['R_max']

    n_tcs = p['n_tcs'] # hill coefficient for TCS
    
    # get logistic growth parameters
    if(log_growth_params is None):
        log_growth_params = {'r': 0.5, 'N_max': 1.0}
    r = log_growth_params['r']
    N_max = log_growth_params['N_max']

    # get growth-dependent parameters
    ## culture OD to carrying capacity ratio
    f = N / N_max
    ## molecular process efficiency
    y = np.maximum(f * (1 - f), 0.0) # clip from below at zero to avoid numerical issues
    y_resource = np.power(y, n_gamma)
    y_rate = np.power(y, n_gamma)
    ## protein degradation rate modulator
    delta = f**n_delta/(1 +  f**n_delta)
    ## effective rate of cell division
    alpha = (1 - f)

    # get total and free resources
    R_total = R_max * y_resource
    R_free = R_total - C_tic

    # get dilution and degradation rates
    d_dil = r * alpha
    d_m = d_m * alpha + d_dil
    d_p = d_p * delta + d_dil

    # get effective rates of translation initiation and elongation
    beta = beta * y_resource
    k_fold = k_fold * (y_rate + b_fold)

    # get translation initiation and elongation rates (for sfGFP)
    r_tl_intn = k_tli_b * M * R_free - k_tli_u * C_tic # translation initiation rate
    r_tl_elng = k_tl * C_tic # trnalsation elongation rate

    # get ODEs
    ODEs = []
    dSdt = - (k_green + b_green) * S + (k_red + b_red) * Sp + k_sp_b * Sp * R - k_sp_u * Rp * S
    ODEs.append(dSdt) # Light sensing module

    dSpdt =  (k_green + b_green) * S - (k_red + b_red) * Sp - k_sp_b * Sp * R + k_sp_u * Rp * S
    ODEs.append(dSpdt) # Transcription output module

    dRdt =  - k_sp_b * Sp * R + k_sp_u * Rp * S
    ODEs.append(dRdt) # Activating complex monomer formation

    dRpdt = k_sp_b * Sp * R - k_sp_u * Rp * S - k_rp_b * Rp * Rp + k_rp_u * Ac
    ODEs.append(dRpdt) # Activating complex monomer formation

    dAcdt = k_rp_b * Rp * Rp - k_rp_u * Ac
    ODEs.append(dAcdt) # Activating complex dimer formation

    dMdt = beta * (Ac**n_tcs / (Kc**n_tcs + Ac**n_tcs) + l0) - d_m * M - r_tl_intn + r_tl_elng
    ODEs.append(dMdt) # Transcription

    dCticdt = r_tl_intn - r_tl_elng
    ODEs.append(dCticdt) # Translation initiation

    dPdt = r_tl_elng - k_fold * P - d_p * P
    ODEs.append(dPdt) # Translation

    dPmdt = k_fold * P - d_p * Pm
    ODEs.append(dPmdt) # Protein folding

    dNdt = r * N * (1 - f)
    ODEs.append(dNdt) # Logistic growth

    return ODEs

# ode integrator
def run_TCS_model(x0, time, params, light, output = 'solve_ivp', log_growth_params = None):
    """
        Run ODE model integration

        Args
            x0: initial condition
            time: time points
            params: model parameters
            light: light condition
            output: output format
            log_growth_params: logistic growth parameters
        Returns:
            solution object from solve_ivp or just the state trajectory
    """

    # integrate the ODE
    sol = scint.solve_ivp(dxdt, [time[0], time[-1]], x0, 
                          args = (params, log_growth_params, light), t_eval = time,
                          method = 'LSODA')

    if not sol.success or sol.y.shape[1] != len(time) or not np.isfinite(sol.y).all():
        raise ValueError(f'ODE integration failed: {sol.message}')

    # Extract results in odeint format OR just return the system states
    if output == 'solve_ivp':
        return sol
    else:
        return sol.y

In [60]:
# DEFINE RESIDUAL FUNCTION FOR CURVE FITTING

def residual(params, replicates):
    all_residuals = []

    for rep in replicates:
        # get x0
        p = params.valuesdict()
        x0 = np.asarray([
            p['S_0'], p['Sp_0'],
            p['R_0'], p['Rp_0'],
            0.0,  # initial active CcaR complex conc. assumed zero
            p['M_0'],
            0.0,    # initial TL initiation complex conc. assumed zero
            p['P_0'], p['Pm_0'],
            rep['log_growth_params']['N0']  # NOTE: N0 comes from the replicate information
        ])

        # integrate ODE model
        xs = run_TCS_model(x0, rep['ts'], params, light=rep['light'], output = 'just y', log_growth_params = rep['log_growth_params'])

        # calculate residual - difference between model and measured fluorescence with each series normalised by its last value
        residual = xs[8,:]/xs[8,-1] - rep['normalised_fluorescence_corr']/rep['normalised_fluorescence_corr'][-1]
        all_residuals.append(residual)

    # return a concatenation of all residuals
    return np.concatenate(all_residuals)

In [61]:
# LOAD EXPERIMENTAL DATA: ONE DICTIONARY PER TECHNICAL WELL AND LIGHT

time_points_to_exclude = [0]

# Current exports are in exp_fits; change this path if they are moved to exp_data.
EXP_DATA_DIR = ROOT / 'Yuhang/26-09-03 Kirill 3/figures_diya_014_analysis/exp_fits'
EXP_FILE_PATTERN = 'GFP_395nm_*.csv'
FLUORESCENCE_COLUMN = 'normalised_fluorescence_corr'

experiment_files = sorted(EXP_DATA_DIR.glob(EXP_FILE_PATTERN))
if not experiment_files:
    raise FileNotFoundError(f'No {EXP_FILE_PATTERN} files found in {EXP_DATA_DIR}')

replicates = []
required_columns = {
    'sample', 'well', 'light', 'hours', 'included', FLUORESCENCE_COLUMN,
    'logistic_r', 'logistic_Nmax', 'logistic_N0',
}
for path in experiment_files:
    data = pd.read_csv(path)
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(f'{path.name}: missing columns {sorted(missing_columns)}')

    for (sample, light, well), readings in data.groupby(
            ['sample', 'light', 'well'], sort=False):
        if light not in ['red', 'green']:
            raise ValueError(f'{path.name}: unknown light {light!r}')
        growth = readings[['logistic_r', 'logistic_Nmax', 'logistic_N0']].drop_duplicates()
        if len(growth) != 1 or not np.isfinite(growth.to_numpy()).all():
            raise ValueError(f'{path.name}, {well}: expected one finite growth parameter set')
        growth = growth.iloc[0]

        # Excluded rows contain NaNs in the exported measurements.
        readings = readings.loc[readings['included'].eq(True)].sort_values('hours')
        ts = readings['hours'].to_numpy(dtype=float)
        fluorescence = readings[FLUORESCENCE_COLUMN].to_numpy(dtype=float)
        if not len(ts) or not (np.isfinite(ts).all() and np.isfinite(fluorescence).all()):
            raise ValueError(f'{path.name}, {well}: empty or nonfinite included measurements')
        if not np.all(np.diff(ts) > 0):
            raise ValueError(f'{path.name}, {well}: time points must be unique and increasing')

        replicates.append({
            'sample': sample,
            'well': well,
            # NOTE: converting timescales into min from h
            'ts': np.delete(ts*60.0,time_points_to_exclude),
            'normalised_fluorescence_corr': np.delete(fluorescence,time_points_to_exclude),
            'light': light,
            # Use the parameter names expected by dxdt, converting timescales into min from h
            'log_growth_params': {
                'r': float(growth['logistic_r']/60),
                'N_max': float(growth['logistic_Nmax']),
                'N0': float(growth['logistic_N0']),
            },
        })

print(f'Loaded {len(replicates)} technical well/light series from {len(experiment_files)} files.')


Loaded 36 technical well/light series from 6 files.


In [62]:
# LOAD DEFAULT GEAGS PARAMETERS - FROM https://doi.org/10.1021/acssynbio.6c00003

parameter_defaults = pd.read_csv(ROOT / 'Yuhang/geags_params_default.csv', index_col=0)
if parameter_defaults['Param name'].duplicated().any():
    raise ValueError('Default parameter names must be unique.')

params = Parameters()
for name, default in parameter_defaults[['Param name', 'Param']].itertuples(index=False, name=None):
    default = float(default)
    if not np.isfinite(default) or default <= 0:
        raise ValueError(f'{name}: expected a finite positive default for multiplicative bounds.')
    params.add(name, value=default, min=default / 1000, max=default * 1000)

params.pretty_print()

Name        Value      Min      Max   Stderr     Vary     Expr Brute_Step
Kc             45    0.045  4.5e+04     None     True     None     None
M_0             1    0.001     1000     None     True     None     None
P_0             1    0.001     1000     None     True     None     None
Pm_0           15    0.015  1.5e+04     None     True     None     None
R_0           180     0.18  1.8e+05     None     True     None     None
R_max           4    0.004     4000     None     True     None     None
Rp_0           30     0.03    3e+04     None     True     None     None
S_0             1    0.001     1000     None     True     None     None
Sp_0            1    0.001     1000     None     True     None     None
b_fold          1    0.001     1000     None     True     None     None
b_green      0.03    3e-05       30     None     True     None     None
b_red         0.8   0.0008      800     None     True     None     None
beta           28    0.028  2.8e+04     None     True     None

In [63]:
# RUN GEAGS FIT: INDEPENDENT PARAMETERS FOR K AND N
MAX_NFEV = 2000
fit_groups = ['K', 'N']
replicates_by_group = {
    group: [rep for rep in replicates if rep['sample'].split('_', 1)[0] == group]
    for group in fit_groups
}
fit_results = {}
best_params_by_group = {}
initial_residuals_by_group = {}

for group, group_replicates in replicates_by_group.items():
    if not group_replicates:
        raise ValueError(f'No experimental replicates found for {group}.')
    # Start both groups independently from the same defaults and bounds.
    fit_params = params.copy()
    for rep in group_replicates:
        if rep['normalised_fluorescence_corr'][-1] <= 0:
            raise ValueError(f"{rep['sample']}, {rep['well']}: final fluorescence must be positive.")
    initial_residuals = residual(fit_params, group_replicates)
    if not np.isfinite(initial_residuals).all():
        raise ValueError(f'{group}: initial model residuals contain NaN or infinity.')
    initial_residuals_by_group[group] = initial_residuals

    fit_result = minimize(
        residual,
        fit_params,
        args=(group_replicates,),
        method='least_squares',
        nan_policy='raise',
        max_nfev=MAX_NFEV,
        x_scale='jac',
    )
    fit_results[group] = fit_result
    best_params_by_group[group] = fit_result.params.copy()
    print(f"{group}: {len(group_replicates)} technical well/light series")
    print(fit_report(fit_result))
    print(f'Converged: {fit_result.success}; {fit_result.message}')
    print(f'Sum of squared residuals: {np.sum(initial_residuals**2):.6g}'
          f' -> {fit_result.chisqr:.6g}')


K: 18 technical well/light series
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 154
    # data points      = 126
    # variables        = 28
    chi-square         = 50.8006350
    reduced chi-square = 0.51837383
    Akaike info crit   = -58.4550047
    Bayesian info crit = 20.9608887
[[Variables]]
    k_green:  0.80183454 +/- 208.061521 (25948.19%) (init = 0.8)
    k_red:    1.49057805 +/- 675.342335 (45307.41%) (init = 1.3)
    b_green:  0.03593213 +/- 21.1585247 (58884.70%) (init = 0.03)
    b_red:    0.81862081 +/- 259.140632 (31655.76%) (init = 0.8)
    k_sp_b:   0.04843517 +/- 1.1017e-04 (0.23%) (init = 0.05)
    k_sp_u:   24.9608401 +/- 5560.34599 (22276.28%) (init = 25)
    k_rp_b:   54.9538750 +/- 0.01525223 (0.03%) (init = 55)
    k_rp_u:   0.11211375 +/- 10.6224242 (9474.68%) (init = 0.08)
    beta:     28.2534424 +/- 3912.08423 (13846.40%) (init = 28)
    l0:       1.0189e-05 +/- 0.00200500 (19678.25%) (init = 1e-05)
    Kc:       50.366

In [65]:
# PLOT SIMULATIONS VS EXPERIMENTAL DATA
COLORS = {'red': 'crimson', 'green': 'darkgreen'}
DASHES = ['solid', 'dashed', 'dotted']
SAVE_HTML = False
OUTPUT_DIR = ROOT / 'Yuhang/26-09-03 Kirill 3/figures_geags_fit'

def new_figure(title, ylabel):
    return figure(title=title, x_axis_label='Time since first OD reading, h',
                  y_axis_label=ylabel, width=400, height=300,
                  tools='pan,wheel_zoom,box_zoom,reset,save')

# Match the legend styling and notebook display used in analyse_diya.ipynb.
def emit(layout, name):
    from bokeh.models import Legend
    for legend in layout.select({'type': Legend}):
        legend.background_fill_color = 'white'
        legend.background_fill_alpha = 1.0
        legend.border_line_color = 'black'
        legend.border_line_alpha = 1.0
        legend.margin = 1
        legend.spacing = 1
        legend.padding = 1
    html_doc = file_html(layout, CDN, 'Plots')
    display(HTML(f'<iframe srcdoc="{html_lib.escape(html_doc)}" width="100%" height="700" style="border:none;"></iframe>'))
    if SAVE_HTML:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        target = OUTPUT_DIR / f'{name}.html'
        target.write_text(file_html(layout, INLINE, name), encoding='utf-8')
        print(f'Saved {target}')

samples = list(dict.fromkeys(rep['sample'] for rep in replicates))
lights = [light for light in ['red', 'green']
          if any(rep['light'] == light for rep in replicates)]
panels = []
for sample in samples:
    group = sample.split('_', 1)[0]
    sample_params = best_params_by_group[group]
    p = sample_params.valuesdict()
    for light in lights:
        sample_replicates = [rep for rep in replicates
                             if rep['sample'] == sample and rep['light'] == light]
        if not sample_replicates:
            continue
        panel = new_figure(f'{sample}, {light}',
                           'Corrected fluorescence / final value')
        for j, rep in enumerate(sample_replicates):
            observed = rep['normalised_fluorescence_corr']
            if observed[-1] <= 0:
                raise ValueError(f"{sample}, {rep['well']}: final fluorescence must be positive.")
            # Normalise each technical replicate by its own final included reading.
            panel.line(rep['ts'] / 60.0, observed / observed[-1],
                       color=COLORS[light], alpha=0.35,
                       line_dash=DASHES[j % 3], legend_label=rep['well'])

            # Use the same initial conditions and mature protein state as residual().
            x0 = np.asarray([
                p['S_0'], p['Sp_0'], p['R_0'], p['Rp_0'], 0.0,
                p['M_0'], 0.0, p['P_0'], p['Pm_0'],
                rep['log_growth_params']['N0'],
            ])
            # Include measurement times in a dense grid for a smooth model curve.
            t_model = np.unique(np.r_[rep['ts'],
                np.linspace(rep['ts'][0], rep['ts'][-1], 200)])
            xs = run_TCS_model(x0, t_model, sample_params, light=light,
                               output='just y', log_growth_params=rep['log_growth_params'])
            predicted = xs[8, :]
            if not np.isfinite(predicted).all() or predicted[-1] <= 0:
                raise ValueError(f"{sample}, {rep['well']}: invalid final model fluorescence.")
            # Same endpoint as this replicate's residual; plot minutes as hours.
            panel.line(t_model / 60.0, predicted / predicted[-1],
                       color=COLORS[light], line_width=3, line_dash='solid',
                       legend_label='Model fit')
        panel.legend.location = 'right'
        panel.legend.click_policy = 'hide'
        panels.append(panel)

if panels:
    emit(gridplot(panels, ncols=len(lights)), 'simulations_vs_experimental_data')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
